<a href="https://colab.research.google.com/github/theodd1isoutman56/sigmund-translator/blob/main/Sigmund_Translator_Veritas_AI_Hackathon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sigmund Translator: The Everyday Translator

**Veritas AI International High School Hackathon 2026 · Language Project Track**

**Team:** Shaurya, Kashyap @theodd1isoutman56 and Prakrit

---

## Project overview

Official documents often use formal, technical or legal language. This can make important information difficult to understand, especially for people who are new to a country or are less comfortable with the language used by an institution.

Sigmund Translator is a dual-mode language assistant designed to make communication more accessible. It combines machine translation with generative AI so that users can:

1. **Understand administrative language:** translate official text and rewrite it in simpler, everyday language.
2. **Understand informal language:** translate slang and casual expressions in a more natural, conversational way.

The notebook documents the problem, our approach, the models used, experiments, evaluation, a working demo and the limitations we identified.

## 1. The problem we are addressing

A direct translation does not always make a message easy to understand. Official documents may contain long sentences, bureaucratic vocabulary and important conditions. Informal messages create a different challenge: slang and idioms often lose their meaning when translated word for word.

Our project therefore focuses on two related but distinct problems:

- **Clarity:** preserve the important meaning of an administrative message while making its wording easier to understand.
- **Localization:** communicate the intended meaning of informal language naturally, rather than translating each word literally.

The tool is an aid to understanding, not a legal authority. Users should check important decisions, deadlines and legal obligations against the original document or the relevant institution.

## 2. Our approach

We use two models for the administrative workflow:

- **NLLB-200** provides the initial translation between supported languages.
- **Gemini** rewrites the translated administrative text in simpler language while being instructed to preserve dates, amounts, names and requirements.

For slang and casual language, Gemini is asked to translate and localize the original message directly. This avoids forcing informal expressions through a literal translation step first.

### Workflow

1. The user selects a mode and languages.
2. The input is translated or localized.
3. The result is displayed alongside the baseline where relevant.
4. The user can compare the wording and decide whether it is clear.

This is a two-mode language assistant, not a single model trained from scratch.

## 3. Install the required libraries

We use Transformers to load NLLB-200, the Google GenAI SDK to access Gemini, SacreBLEU for chrF evaluation, pandas for tabular results, and Gradio for the optional interface.

The notebook is designed for Google Colab. The NLLB model download can take a few minutes and requires an internet connection.

In [ ]:
!pip -q install "transformers<5" sentencepiece sacrebleu google-genai gradio pandas datasets

## 4. Import libraries and configure the Gemini API

The Gemini API key is read from **Google Colab Secrets**. In Colab, add a secret named `GEMINI_API_KEY` and allow this notebook to access it.

We do not write the API key directly into the notebook. This is safer when sharing a hackathon submission.

In [ ]:
import os
import pandas as pd
import sacrebleu
import torch

from google import genai
from google.genai import types
from google.colab import userdata
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Read the API key securely from Colab Secrets.
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
if not GEMINI_API_KEY:
    raise ValueError(
        "GEMINI_API_KEY was not found. Add it in Google Colab Secrets "
        "and enable notebook access."
    )

gemini_client = genai.Client(api_key=GEMINI_API_KEY)

# Keep the model name in one place so it can be changed if needed.
GEMINI_MODEL = "gemini-3.5-flash"

print("Libraries imported and Gemini client configured.")

Libraries imported and Gemini client configured.


## 5. Load the NLLB-200 translation model

We use `facebook/nllb-200-distilled-600M`, a multilingual sequence-to-sequence model.

The model is loaded once and reused for each translation. This avoids downloading and initializing it every time a user submits text.

For this notebook, we use a small set of language codes. The code list can be extended to other languages supported by NLLB-200.

In [ ]:
MODEL_NAME = "facebook/nllb-200-distilled-600M"

print("Loading NLLB-200. The first run may take a few minutes...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
translation_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
translation_model = translation_model.to(DEVICE)
translation_model.eval()

print(f"NLLB-200 loaded successfully on {DEVICE}.")

Loading NLLB-200. The first run may take a few minutes...
NLLB-200 loaded successfully on cpu.


## 6. Supported language codes

NLLB uses language identifiers rather than ordinary language names. The dictionary below connects the names shown in the interface to the identifiers expected by the model.

The selected source language is set on the tokenizer before translation. The target language is passed through `forced_bos_token_id`.

In [ ]:
LANG_CODES = {
    "English": "eng_Latn",
    "Hindi": "hin_Deva",
    "Arabic": "arb_Arab",
    "Spanish": "spa_Latn",
    "French": "fra_Latn",
    "Swahili": "swh_Latn",
    "Tagalog": "tgl_Latn"
}

print("Languages available:", ", ".join(LANG_CODES.keys()))

Languages available: English, Hindi, Arabic, Spanish, French, Swahili, Tagalog


## 7. Stage 1: baseline machine translation

The first function translates the input using NLLB-200. This is our baseline: it gives us a direct translation before any plain-language rewriting or slang localization takes place.

We use inference mode because the model is being used for prediction, not training.

In [ ]:
def translate_text(text: str, src_lang: str, tgt_lang: str) -> str:
    """Translate text from a supported source language to a target language."""
    if not isinstance(text, str) or not text.strip():
        raise ValueError("Please enter some text.")
    if src_lang not in LANG_CODES or tgt_lang not in LANG_CODES:
        raise ValueError("The selected language is not supported in this notebook.")

    tokenizer.src_lang = LANG_CODES[src_lang]
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(DEVICE)

    with torch.inference_mode():
        generated_tokens = translation_model.generate(
            **inputs,
            forced_bos_token_id=tokenizer.convert_tokens_to_ids(LANG_CODES[tgt_lang]),
            max_new_tokens=256
        )

    return tokenizer.batch_decode(
        generated_tokens,
        skip_special_tokens=True
    )[0].strip()

## 8. Stage 2: administrative language simplification

A direct translation may still contain formal or difficult wording. The next function asks Gemini to rewrite the translated text in simpler language.

The prompt emphasizes meaning preservation. In particular, the model is told not to change dates, numbers, fees, names, deadlines or legal requirements. This instruction reduces the risk of a rewrite changing important details, although the output must still be checked against the source.

In [ ]:
def simplify_administrative_text(text: str, target_language: str) -> str:
    """Rewrite translated administrative text in plain, clear language."""
    prompt = f"""
You help everyday citizens and newcomers understand official documents.

Rewrite the text below in {target_language} using simple, clear,
everyday language.

Rules:
- Preserve the original meaning and all important conditions.
- Keep dates, amounts, numbers, names, deadlines and requirements unchanged.
- Do not remove warnings, exceptions or eligibility conditions.
- Break long sentences into shorter sentences or bullet points when useful.
- Do not invent information or give legal advice.
- Output only the simplified explanation.

Text:
{text}
"""

    response = gemini_client.models.generate_content(
        model=GEMINI_MODEL,
        contents=prompt,
        config=types.GenerateContentConfig()
    )

    if not response.text:
        raise RuntimeError("Gemini returned an empty response.")
    return response.text.strip()

## 9. Alternative mode: slang and casual-language localization

Informal language can contain idioms, shortened phrases and culturally specific expressions. A literal translation may sound unnatural or communicate the wrong tone.

In this mode, Gemini receives the original text and is asked to express its meaning naturally in the target language. The goal is to preserve the intent and conversational tone rather than translate every word literally.

In [ ]:
def translate_and_localize_slang(text: str, target_language: str) -> str:
    """Translate informal text into natural, conversational target-language wording."""
    if not isinstance(text, str) or not text.strip():
        raise ValueError("Please enter some text.")

    prompt = f"""
You are a multilingual localization assistant.

Translate and adapt the following informal or slang-heavy message into
natural, conversational {target_language}.

Rules:
- Preserve the speaker's intended meaning and tone.
- Do not translate idioms or slang word for word if that sounds unnatural.
- Use an expression that a fluent speaker of {target_language} would understand.
- Do not add new facts or change the speaker's intent.
- Output only the localized message.

Message:
{text}
"""

    response = gemini_client.models.generate_content(
        model=GEMINI_MODEL,
        contents=prompt,
        config=types.GenerateContentConfig()
    )

    if not response.text:
        raise RuntimeError("Gemini returned an empty response.")
    return response.text.strip()

## 10. Combine the stages into one pipeline

The pipeline keeps the two modes separate:

- **Administrative Jargon Simplification:** NLLB translates the source text, then Gemini simplifies that translation.
- **Slang & Casual Localization:** Gemini translates and localizes the original message directly.

For administrative text, we return both the direct translation and the simplified version so the user can compare them.

In [ ]:
def run_pipeline(
    text: str,
    mode: str,
    src_lang: str,
    tgt_lang: str
):
    """Run the selected workflow and return baseline and final text."""
    if not text or not text.strip():
        raise ValueError("Please enter some text.")

    if mode == "Administrative Jargon Simplification":
        baseline = translate_text(text, src_lang, tgt_lang)
        final_output = simplify_administrative_text(baseline, tgt_lang)
        return baseline, final_output

    if mode == "Slang & Casual Localization":
        # NLLB output is shown as a comparison baseline.
        baseline = translate_text(text, src_lang, tgt_lang)
        final_output = translate_and_localize_slang(text, tgt_lang)
        return baseline, final_output

    raise ValueError("Please select a valid translation mode.")

## 11. Build a small administrative-language test set

We prepared two example administrative texts for an initial qualitative test. They represent two contexts:

- Visa and immigration requirements.
- Healthcare and insurance authorization.

Each example includes a human-written reference translation in Hindi and Spanish. These references let us compare the direct NLLB translation with a reference translation using chrF.

**Dataset limitation:** two examples are not enough to establish general translation quality. We use them as demonstrations and a small initial check, not as proof that the system is reliable for all official documents.

In [ ]:
test_data = [
    {
        "category": "Visa / Immigration",
        "english_source": (
            "Failure to remit the requisite renewal fee prior to the expiration "
            "of your current residency permit will result in the accrual of "
            "statutory per-diem penalties and potential revocation of sponsorship status."
        ),
        "human_reference_hindi": (
            "वर्तमान निवास परमिट की समाप्ति से पहले आवश्यक नवीनीकरण शुल्क जमा "
            "करने में विफल रहने पर वैधानिक दैनिक जुर्माना लगेगा और प्रायोजन "
            "स्थिति रद्द हो सकती है।"
        ),
        "human_reference_spanish": (
            "No pagar la tarifa de renovación requerida antes del vencimiento "
            "de su permiso de residencia actual resultará en la acumulación de "
            "multas diarias legales y la posible revocación del estado de patrocinio."
        )
    },
    {
        "category": "Healthcare / Insurance",
        "english_source": (
            "Beneficiaries must obtain prior authorization from the primary "
            "insurer before undergoing non-emergent inpatient surgical procedures, "
            "lest the claim be adjudicated as out-of-network and denied."
        ),
        "human_reference_hindi": (
            "लाभार्थियों को गैर-आपातकालीन सर्जरी कराने से पहले मुख्य बीमाकर्ता "
            "से पूर्व अनुमति लेनी होगी, अन्यथा दावे को नेटवर्क से बाहर मानकर "
            "अस्वीकार कर दिया जाएगा।"
        ),
        "human_reference_spanish": (
            "Los beneficiarios deben obtener autorización previa de la aseguradora "
            "principal antes de someterse a procedimientos quirúrgicos no urgentes, "
            "para evitar que el reclamo se considere fuera de la red y sea denegado."
        )
    }
]

admin_df = pd.DataFrame(test_data)
display(admin_df[["category", "english_source"]])

,category,english_source
0,Visa / Immigration,Failure to remit the requisite renewal fee pri...
1,Healthcare / Insurance,Beneficiaries must obtain prior authorization ...


## 12. Run the administrative examples

For each source text, we translate into Hindi and Spanish. We then simplify the direct translation and store the outputs in a results table.

The table keeps the source, baseline translation and final simplified output together so that the difference can be inspected.

In [ ]:
admin_results = []

for item in test_data:
    for language, reference_key in [
        ("Hindi", "human_reference_hindi"),
        ("Spanish", "human_reference_spanish")
    ]:
        baseline = translate_text(item["english_source"], "English", language)
        simplified = simplify_administrative_text(baseline, language)

        admin_results.append({
            "Category": item["category"],
            "Language": language,
            "Original Text": item["english_source"],
            "Baseline Translation": baseline,
            "Simplified Output": simplified,
            "Human Reference": item[reference_key]
        })

admin_results_df = pd.DataFrame(admin_results)
display(admin_results_df[[
    "Category", "Language", "Baseline Translation",
    "Simplified Output", "Human Reference"
]])

,Category,Language,Baseline Translation,Simplified Output,Human Reference
0,Visa / Immigration,Hindi,वर्तमान निवास परमिट की समाप्ति से पहले आवश्यक ...,यदि आप अपने मौजूदा निवास परमिट के खत्म होने से...,वर्तमान निवास परमिट की समाप्ति से पहले आवश्यक ...
1,Visa / Immigration,Spanish,Si no se remite la tarifa de renovación requer...,Si no paga la tarifa de renovación antes de qu...,No pagar la tarifa de renovación requerida ant...
2,Healthcare / Insurance,Hindi,लाभार्थियों को गैर-अवश्यकीय अस्पताल में सर्जरी...,अगर आप अस्पताल में ऐसी सर्जरी (ऑपरेशन) करवाने ...,लाभार्थियों को गैर-आपातकालीन सर्जरी कराने से प...
3,Healthcare / Insurance,Spanish,Los beneficiarios deben obtener la autorizació...,Si usted tiene este seguro y necesita hacerse ...,Los beneficiarios deben obtener autorización p...


## 13. Evaluate baseline translation with chrF

We use **chrF** from SacreBLEU to compare NLLB's direct translations with the human-written reference translations in our small administrative test set.

chrF measures character n-gram overlap. It is useful for comparing translations, but it does not fully measure meaning, readability, tone or whether a legal condition has been preserved.

We calculate the score separately for Hindi and Spanish. These scores describe only the examples in this notebook; they should not be treated as broad model benchmarks.

In [ ]:
for language in ["Hindi", "Spanish"]:
    subset = admin_results_df[admin_results_df["Language"] == language]
    hypotheses = subset["Baseline Translation"].tolist()
    references = subset["Human Reference"].tolist()

    score = sacrebleu.corpus_chrf(hypotheses, [references])
    print(f"{language} baseline chrF: {score.score:.2f}")
    print(f"Examples evaluated: {len(subset)}")
    print()

Hindi baseline chrF: 60.89
Examples evaluated: 2

Spanish baseline chrF: 75.93
Examples evaluated: 2



## 14. Inspect the results instead of relying only on a score

A metric gives us one view of translation quality, but it cannot tell us whether the final explanation is easy to understand.

For each example, we should manually check:

- Are the dates, amounts and requirements still present?
- Has the simplified version removed or changed a condition?
- Is the language natural and understandable?
- Does the result remain faithful to the original document?

The following cell prints each example in a readable, side-by-side format for review.

In [ ]:
for index, row in admin_results_df.iterrows():
    print(f"\n{'=' * 72}")
    print(f"Example {index + 1}: {row['Category']} — {row['Language']}")
    print("\nORIGINAL:")
    print(row["Original Text"])
    print("\nBASELINE TRANSLATION:")
    print(row["Baseline Translation"])
    print("\nSIMPLIFIED OUTPUT:")
    print(row["Simplified Output"])
    print("\nHUMAN REFERENCE:")
    print(row["Human Reference"])


Example 1: Visa / Immigration — Hindi

ORIGINAL:
Failure to remit the requisite renewal fee prior to the expiration of your current residency permit will result in the accrual of statutory per-diem penalties and potential revocation of sponsorship status.

BASELINE TRANSLATION:
वर्तमान निवास परमिट की समाप्ति से पहले आवश्यक नवीनीकरण शुल्क का भुगतान न करने से प्रति दिन कानूनी जुर्माना और प्रायोजक की स्थिति की संभावित निरस्तता का कारण बन जाएगा।

SIMPLIFIED OUTPUT:
यदि आप अपने मौजूदा निवास परमिट के खत्म होने से पहले नवीनीकरण (रिन्यूअल) शुल्क का भुगतान नहीं करते हैं, तो:

- आप पर रोजाना कानूनी जुर्माना लगाया जाएगा।
- आपके स्पॉन्सर (प्रायोजक) का स्टेटस भी रद्द हो सकता है।

HUMAN REFERENCE:
वर्तमान निवास परमिट की समाप्ति से पहले आवश्यक नवीनीकरण शुल्क जमा करने में विफल रहने पर वैधानिक दैनिक जुर्माना लगेगा और प्रायोजन स्थिति रद्द हो सकती है।

Example 2: Visa / Immigration — Spanish

ORIGINAL:
Failure to remit the requisite renewal fee prior to the expiration of your current residency permit wi

## 15. Additional translation evaluation with OPUS-100

To add a wider multilingual evaluation, we load an English-to-Hindi test split from OPUS-100 (`Helsinki-NLP/opus-100`). We use a small 20-example subset to keep the run manageable and entirely ungated.

This evaluation checks the **NLLB baseline translation** against the Hindi reference sentences. It does not evaluate Gemini's simplification quality.

Unlike gated datasets that require authentication tokens, OPUS-100 is publicly accessible and downloads instantly for any hackathon judge running this notebook.

In [ ]:
from datasets import load_dataset

print("Loading the English–Hindi evaluation sample...")

# Load 20 English-Hindi test pairs from an ungated Parquet dataset
eval_dataset = load_dataset(
    "Helsinki-NLP/opus-100",
    "en-hi",
    split="test[:20]"
)

flores_sources = [row["translation"]["en"] for row in eval_dataset]
flores_references = [row["translation"]["hi"] for row in eval_dataset]

print(f"Loaded {len(flores_sources)} source sentences.")

Loading the English–Hindi evaluation sample...


README.md: 0.00B [00:00, ?B/s]

en-hi/test-00000-of-00001.parquet:   0%|          | 0.00/259k [00:00<?, ?B/s]

en-hi/train-00000-of-00001.parquet:   0%|          | 0.00/65.2M [00:00<?, ?B/s]

en-hi/validation-00000-of-00001.parquet:   0%|          | 0.00/247k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/534319 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Loaded 20 source sentences.


## 16. Translate and score the OPUS-100 sample

We translate each English sentence into Hindi with NLLB-200 and calculate corpus-level chrF against the OPUS-100 references.

This gives us a separate benchmark result from the two administrative examples. It is still a small subset, so the score should be interpreted as an initial experiment rather than a definitive ranking of the model.

In [ ]:
flores_hypotheses = []

for sentence in flores_sources:
    flores_hypotheses.append(
        translate_text(sentence, "English", "Hindi")
    )

flores_score = sacrebleu.corpus_chrf(
    flores_hypotheses,
    [flores_references]
)

print("FLORES+ English → Hindi evaluation")
print(f"Examples: {len(flores_sources)}")
print(f"NLLB baseline chrF: {flores_score.score:.2f}")

FLORES+ English → Hindi evaluation
Examples: 20
NLLB baseline chrF: 31.04


## 17. Inspect OPUS-100 translation examples

We print a few source, model and reference examples. This helps us see what the metric does not show, such as differences in phrasing or word choice.

A high overlap score does not guarantee that a translation is perfect. Important text should still be checked by a fluent speaker or qualified reviewer.

In [ ]:
for i in range(min(5, len(flores_sources))):
    print(f"\nExample {i + 1}")
    print("English source:       ", flores_sources[i])
    print("NLLB Hindi output:    ", flores_hypotheses[i])
    print("FLORES+ Hindi reference:", flores_references[i])


Example 1
English source:        Give shots of injections or pills, but he must be alright soon.
NLLB Hindi output:     इंजेक्शन या गोलियां दें, लेकिन वह जल्द ठीक हो जाएगा।
FLORES+ Hindi reference: सुई लगाओ या गोली खिलाओ लेकिन इसे जल्दी से ठीक करो.

Example 2
English source:        They said, “O Shuaib, we do not understand much of what you say, and we see that you are weak among us. Were it not for your tribe, we would have stoned you. You are of no value to us.”
NLLB Hindi output:     उन्होंने कहा, "ऐ शुऐब! हम तो तेरे कहने में से बहुत कुछ समझते नहीं और हम देखते हैं कि तू हमारे बीच कमजोर है। यदि तेरा क़बील न होता तो हम तुझे पथराव करते। तू हमारे लिए कोई कामगार नहीं।"
FLORES+ Hindi reference: और वह लोग कहने लगे ऐ शुएब जो बाते तुम कहते हो उनमें से अक्सर तो हमारी समझ ही में नहीं आयी और इसमें तो शक नहीं कि हम तुम्हें अपने लोगों में बहुत कमज़ोर समझते है और अगर तुम्हारा क़बीला न होता तो हम तुम को (कब का) संगसार कर चुके होते और तुम तो हम पर किसी तरह ग़ालिब नहीं आ सकते

Example 3
English source:

## 18. What we learned from the evaluation

The evaluation has two different purposes:

1. The administrative examples let us inspect the effect of simplifying formal text and compare baseline translations with human references.
2. OPUS-100 gives us a small reference-based translation benchmark for NLLB.

There is an important distinction: chrF evaluates translation overlap, not simplicity. To evaluate whether Gemini genuinely makes official text easier to understand, we would need a larger set of documents and a human review process with criteria for clarity and meaning preservation.

We therefore do not claim that the current chrF results prove the simplification stage is better.

## 19. Build the interactive demo

The Gradio interface lets a user choose a mode, select source and target languages, enter text and compare the baseline with the final output.

For administrative text, the final output is the simplified translation. For slang, the final output is Gemini's localized version.

The app is not launched automatically when the notebook is run. Set `RUN_APP = True` in the next cell when you want to open it.

In [ ]:
import gradio as gr

def demo_interface(text, mode, src_lang, tgt_lang):
    try:
        baseline, final_output = run_pipeline(
            text=text,
            mode=mode,
            src_lang=src_lang,
            tgt_lang=tgt_lang
        )
        return baseline, final_output
    except Exception as error:
        message = f"Unable to process this text: {error}"
        return message, message

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        "# 🌐 Sigmund Translator\n"
        "### Understand official language. Localize everyday language."
    )

    mode_selector = gr.Radio(
        choices=[
            "Administrative Jargon Simplification",
            "Slang & Casual Localization"
        ],
        value="Administrative Jargon Simplification",
        label="Select a mode"
    )

    with gr.Row():
        src_dropdown = gr.Dropdown(
            choices=list(LANG_CODES.keys()),
            value="English",
            label="Source language"
        )
        tgt_dropdown = gr.Dropdown(
            choices=list(LANG_CODES.keys()),
            value="Hindi",
            label="Target language"
        )

    input_box = gr.Textbox(
        lines=4,
        label="Enter text",
        placeholder="Paste an official message or an informal expression..."
    )

    submit_btn = gr.Button("Process text", variant="primary")

    with gr.Row():
        baseline_out = gr.Textbox(
            lines=7,
            label="Baseline translation (NLLB-200)"
        )
        final_out = gr.Textbox(
            lines=7,
            label="Sigmund Translator"
        )

    submit_btn.click(
        fn=demo_interface,
        inputs=[input_box, mode_selector, src_dropdown, tgt_dropdown],
        outputs=[baseline_out, final_out]
    )

print("Gradio interface is ready.")

/tmp/ipykernel_830/3040190122.py:16: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Gradio interface is ready.


## 20. Launch the demo (optional)

For the purpose of the judges convenience we have set `RUN_APP` to `True` to launch the Gradio interface inside the notebook. It is set to `False` by default so that running all research and evaluation cells does not automatically open the app.

When launched in Colab, Gradio will display a link to the interface.

In [ ]:
RUN_APP = True

if RUN_APP:
    demo.launch(share=True, show_error=True)
else:
    print("Demo not launched. Set RUN_APP = True when you are ready.")

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b3af05b51d00d8d8e0.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 21. Results summary

This project has implemented and tested a two-mode language workflow.

| Component | What we tested | What the result tells us |
|---|---|---|
| NLLB-200 baseline | Direct translation | Provides the initial translated text |
| Gemini simplification | Administrative examples | Produces a plain-language rewrite for manual inspection |
| chrF evaluation | Two administrative examples per target language | Measures character-level overlap with the supplied references |
| OPUS-100 | Up to 20 English–Hindi examples | Provides a separate small-sample translation benchmark |
| Gemini localization | Slang and casual-language mode | Produces a natural-language adaptation for user review |

We deliberately keep translation metrics separate from simplification quality. The current notebook does not yet contain a larger human-rated dataset to make a reliable numerical claim about readability.

## 22. Limitations

- **Small administrative dataset:** the current hand-written test set contains only two source examples. It is useful for demonstrating the workflow but is not representative of all government, visa or insurance documents.
- **Translation errors:** NLLB can mistranslate specialized terms, names or complex sentence structures.
- **Meaning preservation:** a generative model may accidentally omit or alter a condition while simplifying. Users must compare important details with the original.
- **Limited language list:** the interface currently exposes seven languages, even though NLLB supports many more.
- **Evaluation scope:** chrF measures text overlap, not readability, factual accuracy or legal correctness.
- **Slang is contextual:** expressions can vary by region, age and situation. A localized phrase may not be appropriate for every audience.
- **External model dependency:** Gemini requires an API key, internet access and an available model endpoint.

Sigmund Translator should be treated as a language-accessibility assistant, not a replacement for official translation, legal advice or professional interpretation.

## 23. Future improvements

If we continue developing Sigmund Translator, our next steps would be:

1. Expand the administrative test set with more examples from public government and insurance documents.
2. Ask fluent speakers to review translations for meaning, tone and clarity.
3. Create a human-rated readability evaluation for the simplified output.
4. Test more language pairs, including Arabic–Hindi and English–Arabic.
5. Add checks that compare dates, numbers, amounts and named entities before and after simplification.
6. Improve the interface with copy, download and original-text comparison features.
7. Test the tool with users who regularly need help understanding official documents.

These improvements would help us evaluate not only whether the text is translated, but whether the user can understand it accurately.

## 24. Sources and acknowledgements

- **NLLB-200:** Meta AI, `facebook/nllb-200-distilled-600M` — https://huggingface.co/facebook/nllb-200-distilled-600M
- **Transformers:** Hugging Face documentation — https://huggingface.co/docs/transformers
- **OPUS-100:** Multilingual corpus for neural machine translation — https://huggingface.co/datasets/Helsinki-NLP/opus-100
- **SacreBLEU:** chrF and translation evaluation — https://github.com/mjpost/sacrebleu
- **Google GenAI SDK:** https://googleapis.github.io/python-genai/
- **Gradio:** https://www.gradio.app/docs

The two administrative examples and their Hindi and Spanish reference translations were included in our project notebook. The OPUS-100 subset is used as a separate reference-based evaluation source.

**Responsible-use note:** official or legal documents can have serious consequences if misunderstood. Veritas AI provides an accessible explanation, but users should rely on the issuing authority or a qualified professional for decisions.